# Angular 13 — Complete Reference Guide
### Detailed Notes | Examples | Use Cases | Interview Q&A

> **Angular 13** was released on **November 3, 2021**. It is a landmark release that made **Ivy the only compilation engine**, dropped IE11 support, and introduced several developer-experience improvements.

---

## Table of Contents
1. [Detailed Notes](#detailed-notes) — What changed and why
2. [Code Examples](#examples) — Hands-on code for every feature
3. [Use Cases](#use-cases) — Real-world application scenarios
4. [Interview Q&A](#interview-qa) — 20+ interview questions with answers

---

## Version Requirements

| Dependency | Required Version |
|---|---|
| Node.js | `^12.20.0 \|\| ^14.15.0 \|\| >=16.10.0` |
| TypeScript | `4.4.x` |
| RxJS | `^7.4.0` |
| Angular CLI | `13.x` |

```bash
# Upgrade command
ng update @angular/core@13 @angular/cli@13
```

# Section 1 — Detailed Notes

---

## 1.1 Ivy is Now the Only Rendering Engine

### What is Ivy?
Ivy is Angular's **compilation and rendering pipeline**, introduced as opt-in in Angular 8 and made default in Angular 9. Angular 13 **permanently removed View Engine** — there is no flag to switch back.

### What Ivy Enables
- **Locality principle**: Each component is compiled independently. Only the component file itself is needed to compile it (no global knowledge of other components).
- **Tree-shakable**: Unused Angular features are removed from the final bundle.
- **Incremental DOM**: Instead of a Virtual DOM, Ivy uses instructions that update the real DOM directly — lower memory overhead.
- **Better debugging**: `ng.getComponent()`, `ng.applyChanges()` APIs available in DevTools console.

### Impact on Library Authors
Before Angular 13, libraries had to ship pre-compiled **View Engine metadata** (`metadata.json`) for compatibility. Angular 13 removes this — libraries only need to ship Ivy-compatible output.

```
Angular Package Format (APF) v13:
  ✅ Ships ES2020 + ESM (modern JS modules)
  ✅ No UMD bundles required (reduces package size)
  ✅ Partial compilation for maximum compatibility
  ❌ Removed: View Engine metadata (.metadata.json files)
```

### Partial Compilation (`compilationMode: 'partial'`)
Library authors use **partial compilation** so their pre-compiled output works across different Angular versions:

```json
// tsconfig.lib.prod.json (in a library project)
{
  "angularCompilerOptions": {
    "compilationMode": "partial"
  }
}
```

---

## 1.2 IE11 Support Dropped

### Why IE11 Was Dropped
- IE11 market share fell below 1% globally
- Maintaining IE11 compatibility required `zone.js` patches, ES5 polyfills, and differential loading
- Prevented use of modern APIs (Intersection Observer, CSS Custom Properties, Fetch API)

### What Was Removed
| Removed | Reason |
|---|---|
| ES5 build output | IE11 needs ES5; modern browsers support ES2017+ |
| Differential loading | No longer needed without ES5 target |
| IE-specific polyfills | `core-js`, `classlist.js`, `web-animations-js` |
| `zone.js` IE patches | IE-specific async patching removed |

### Build Target Changed
```json
// tsconfig.json — Angular 13 default
{
  "compilerOptions": {
    "target": "ES2017"   // was ES5 in older versions
  }
}
```

### Polyfills Cleanup
```typescript
// polyfills.ts — BEFORE (Angular 12 with IE11)
import 'core-js/es/symbol';
import 'core-js/es/object';
import 'core-js/es/function';
import 'core-js/es/array';
import 'classlist.js';
import 'web-animations-js';
import 'zone.js';

// polyfills.ts — AFTER (Angular 13, no IE11)
import 'zone.js';   // only zone.js needed
```

---

## 1.3 Simplified Dynamic Component Creation

### The Old API (Deprecated in v13)
Creating dynamic components required a **ComponentFactoryResolver** to get a factory first:

```typescript
// BEFORE Angular 13 — 4 steps needed
// Step 1: Import ComponentFactoryResolver
constructor(
  private resolver: ComponentFactoryResolver,
  private vcr: ViewContainerRef
) {}

// Step 2: Resolve the factory
const factory = this.resolver.resolveComponentFactory(AlertComponent);

// Step 3: Create component using the factory
const componentRef = this.vcr.createComponent(factory);

// Step 4: Set inputs
componentRef.instance.message = 'Hello!';
```

### The New API (Angular 13+)
Pass the component **class directly**:

```typescript
// AFTER Angular 13 — 2 steps
constructor(private vcr: ViewContainerRef) {}

// Step 1: Create directly (no factory needed!)
const componentRef = this.vcr.createComponent(AlertComponent);

// Step 2: Set inputs
componentRef.instance.message = 'Hello!';
```

`ComponentFactoryResolver` still exists but is **deprecated** and will be removed in a future version.

---

## 1.4 Persistent Build Cache (Enabled by Default)

Angular CLI 13 enables **disk-based caching** for builds by default.

### How It Works
- Build artifacts are stored in `.angular/cache/` folder
- On subsequent builds, unchanged files are restored from cache
- Only changed files and their dependents are recompiled

### Configuration
```json
// angular.json
{
  "cli": {
    "cache": {
      "enabled": true,           // default: true
      "path": ".angular/cache",  // default path
      "environment": "all"       // "ci" | "local" | "all"
    }
  }
}
```

### Adding Cache to `.gitignore`
```gitignore
# Angular build cache
.angular/cache
```

### CI/CD Benefit
Cache can be persisted between CI runs to speed up pipelines:
```yaml
# GitHub Actions example
- name: Cache Angular build
  uses: actions/cache@v3
  with:
    path: .angular/cache
    key: angular-cache-${{ hashFiles('package-lock.json') }}
```

---

## 1.5 TypeScript 4.4 Support

### Key TypeScript 4.4 Features Used in Angular

#### A. Control Flow Analysis of Aliased Conditions
```typescript
// TypeScript 4.4 — smarter narrowing
function processUser(user: User | null) {
  const hasUser = user !== null;  // aliased condition

  if (hasUser) {
    console.log(user.name);  // ✅ TypeScript 4.4 knows user is not null here
  }
}
```

#### B. `--useUnknownInCatchVariables`
```typescript
// TypeScript 4.4 with strictness enabled
try {
  fetchData();
} catch (err) {
  // err is now 'unknown' instead of 'any'
  if (err instanceof Error) {
    console.error(err.message);  // safe
  }
}
```

---

## 1.6 RxJS 7.4 Support

### Key Improvements in RxJS 7.x vs 6.x
| Feature | RxJS 6 | RxJS 7.4 |
|---|---|---|
| Bundle size | ~60KB | ~47KB (25% smaller) |
| TypeScript types | Good | Improved (stricter) |
| `switchMap` error handling | Manual | Better inference |
| `animationFrames()` | Not available | New |
| `firstValueFrom` / `lastValueFrom` | Not available | Replaces `.toPromise()` |

### Migration: `toPromise()` → `firstValueFrom` / `lastValueFrom`
```typescript
// OLD (deprecated in RxJS 7)
const user = await this.userService.getUser(1).toPromise();

// NEW (RxJS 7+)
import { firstValueFrom, lastValueFrom } from 'rxjs';

const user = await firstValueFrom(this.userService.getUser(1));
```

---

## 1.7 TestBed Automatic Teardown

Before Angular 13, `TestBed` kept the DOM and module alive between tests, causing memory leaks and test pollution.

### New Default Behavior
```typescript
// Angular 13 default — teardown is enabled automatically
TestBed.configureTestingModule({
  declarations: [MyComponent],
  teardown: { destroyAfterEach: true }  // this is now the DEFAULT
});
```

### Opting Out (for gradual migration)
```typescript
// If your tests break, opt out temporarily:
TestBed.configureTestingModule({
  declarations: [MyComponent],
  teardown: { destroyAfterEach: false }  // opt out
});
```

---

## 1.8 `DATE_PIPE_DEFAULT_OPTIONS` Token

Angular 13 adds a DI token to configure `DatePipe` globally:

```typescript
// app.module.ts or bootstrapApplication providers
providers: [
  {
    provide: DATE_PIPE_DEFAULT_OPTIONS,
    useValue: { dateFormat: 'longDate', timezone: 'UTC' }
  }
]
```

# Section 2 — Code Examples

---

## Example 1: Dynamic Component Creation (Before vs After)

### Scenario: A Toast Notification Service

#### OLD WAY — Angular 12 and below
```typescript
// toast.service.ts (Angular < 13)
import {
  Injectable,
  ComponentFactoryResolver,
  ApplicationRef,
  Injector,
  EmbeddedViewRef
} from '@angular/core';
import { ToastComponent } from './toast.component';

@Injectable({ providedIn: 'root' })
export class ToastService {
  constructor(
    private resolver: ComponentFactoryResolver,
    private appRef: ApplicationRef,
    private injector: Injector
  ) {}

  show(message: string): void {
    // 1. Resolve factory (boilerplate)
    const factory = this.resolver.resolveComponentFactory(ToastComponent);

    // 2. Create component
    const componentRef = factory.create(this.injector);

    // 3. Set input
    componentRef.instance.message = message;

    // 4. Attach to app
    this.appRef.attachView(componentRef.hostView);
    const domElem = (componentRef.hostView as EmbeddedViewRef<any>).rootNodes[0];
    document.body.appendChild(domElem);

    // 5. Auto-destroy after 3s
    setTimeout(() => {
      this.appRef.detachView(componentRef.hostView);
      componentRef.destroy();
    }, 3000);
  }
}
```

#### NEW WAY — Angular 13+
```typescript
// toast.service.ts (Angular 13+)
import { Injectable, ApplicationRef, createComponent, EnvironmentInjector } from '@angular/core';
import { ToastComponent } from './toast.component';

@Injectable({ providedIn: 'root' })
export class ToastService {
  constructor(
    private appRef: ApplicationRef,
    private injector: EnvironmentInjector
  ) {}

  show(message: string): void {
    // 1. Create component directly — NO FACTORY!
    const componentRef = createComponent(ToastComponent, {
      environmentInjector: this.injector
    });

    // 2. Set input
    componentRef.instance.message = message;

    // 3. Attach to app
    this.appRef.attachView(componentRef.hostView);
    document.body.appendChild(componentRef.location.nativeElement);

    // 4. Auto-destroy after 3s
    setTimeout(() => {
      this.appRef.detachView(componentRef.hostView);
      componentRef.destroy();
    }, 3000);
  }
}
```

```typescript
// toast.component.ts
@Component({
  selector: 'app-toast',
  template: `
    <div class="toast">
      <span>{{ message }}</span>
    </div>
  `,
  styles: [`
    .toast {
      position: fixed;
      bottom: 20px;
      right: 20px;
      background: #333;
      color: white;
      padding: 12px 24px;
      border-radius: 4px;
      z-index: 9999;
    }
  `]
})
export class ToastComponent {
  @Input() message = '';
}
```

```typescript
// Usage anywhere in the app
@Component({ selector: 'app-root', template: `<button (click)="notify()">Notify</button>` })
export class AppComponent {
  constructor(private toast: ToastService) {}
  notify() { this.toast.show('File saved successfully!'); }
}
```

---

## Example 2: ViewContainerRef.createComponent() with Inputs

```typescript
// dynamic-host.component.ts
@Component({
  selector: 'app-dynamic-host',
  template: `<ng-container #host></ng-container>`
})
export class DynamicHostComponent implements OnInit {
  @ViewChild('host', { read: ViewContainerRef }) host!: ViewContainerRef;

  constructor() {}

  ngOnInit(): void {
    this.loadWidget('chart', { data: [10, 20, 30], title: 'Sales' });
  }

  loadWidget(type: string, config: any): void {
    this.host.clear();

    // Angular 13 — direct class reference, no factory resolver
    const componentMap: Record<string, any> = {
      chart: ChartWidgetComponent,
      table: TableWidgetComponent,
      map:   MapWidgetComponent
    };

    const ComponentClass = componentMap[type];
    if (!ComponentClass) return;

    const ref = this.host.createComponent(ComponentClass);

    // Set inputs directly on the instance
    ref.instance.data  = config.data;
    ref.instance.title = config.title;
    ref.changeDetectorRef.detectChanges();
  }
}
```

---

## Example 3: RxJS 7.4 — `firstValueFrom` and `lastValueFrom`

```typescript
// user.service.ts
import { Injectable } from '@angular/core';
import { HttpClient } from '@angular/common/http';
import { firstValueFrom, lastValueFrom } from 'rxjs';
import { retry, timeout } from 'rxjs/operators';

export interface User { id: number; name: string; email: string; }

@Injectable({ providedIn: 'root' })
export class UserService {
  private api = 'https://jsonplaceholder.typicode.com';

  constructor(private http: HttpClient) {}

  // Observable approach
  getUser$(id: number) {
    return this.http.get<User>(`${this.api}/users/${id}`).pipe(
      retry(3),
      timeout(5000)
    );
  }

  // Promise approach using firstValueFrom (Angular 13+ recommended)
  async getUser(id: number): Promise<User> {
    return firstValueFrom(this.getUser$(id));
  }

  // lastValueFrom — waits for the observable to COMPLETE, takes the last emission
  async searchUsers(query: string): Promise<User[]> {
    const results$ = this.http.get<User[]>(`${this.api}/users?q=${query}`);
    return lastValueFrom(results$);
  }
}
```

```typescript
// profile.component.ts
@Component({
  template: `
    <div *ngIf="user">
      <h2>{{ user.name }}</h2>
      <p>{{ user.email }}</p>
    </div>
    <p *ngIf="error">{{ error }}</p>
  `
})
export class ProfileComponent implements OnInit {
  user?: User;
  error?: string;

  constructor(private userService: UserService) {}

  async ngOnInit(): Promise<void> {
    try {
      this.user = await this.userService.getUser(1);
    } catch (err: unknown) {  // TypeScript 4.4 — unknown catch type
      if (err instanceof Error) {
        this.error = err.message;
      }
    }
  }
}
```

---

## Example 4: TestBed with Automatic Teardown

```typescript
// user.component.spec.ts
import { ComponentFixture, TestBed } from '@angular/core/testing';
import { UserComponent } from './user.component';
import { UserService } from './user.service';
import { of } from 'rxjs';

describe('UserComponent', () => {
  let component: UserComponent;
  let fixture: ComponentFixture<UserComponent>;
  let mockUserService: jasmine.SpyObj<UserService>;

  beforeEach(async () => {
    // Create a spy for UserService
    mockUserService = jasmine.createSpyObj('UserService', ['getUser$']);
    mockUserService.getUser$.and.returnValue(of({ id: 1, name: 'Alice', email: 'alice@test.com' }));

    await TestBed.configureTestingModule({
      declarations: [UserComponent],
      providers: [
        { provide: UserService, useValue: mockUserService }
      ]
      // Angular 13: teardown: { destroyAfterEach: true } is now DEFAULT
      // No need to specify — DOM is cleaned up automatically after each test
    }).compileComponents();

    fixture = TestBed.createComponent(UserComponent);
    component = fixture.componentInstance;
    fixture.detectChanges();
  });

  it('should display user name', () => {
    expect(fixture.nativeElement.querySelector('h2').textContent).toBe('Alice');
  });

  it('should call getUser$ on init', () => {
    expect(mockUserService.getUser$).toHaveBeenCalledWith(1);
  });

  // ✅ No need to manually destroy! Angular 13 TestBed handles cleanup automatically
});
```

---

## Example 5: Persistent Build Cache Configuration

```json
// angular.json — production configuration with cache
{
  "$schema": "./node_modules/@angular/cli/lib/config/schema.json",
  "version": 1,
  "cli": {
    "cache": {
      "enabled": true,
      "path": ".angular/cache",
      "environment": "all"
    }
  },
  "projects": {
    "my-app": {
      "architect": {
        "build": {
          "builder": "@angular-devkit/build-angular:browser",
          "options": {
            "outputPath": "dist/my-app",
            "index": "src/index.html",
            "main": "src/main.ts"
          }
        }
      }
    }
  }
}
```

```bash
# First build (no cache)
ng build --configuration=production
# Build time: ~45 seconds

# Second build (with cache, no changes)
ng build --configuration=production
# Build time: ~3 seconds  ← 93% faster!

# Disable cache for a single run
ng build --no-cache

# Clear cache manually
npx ng cache clean
# OR
rm -rf .angular/cache
```

---

## Example 6: DATE_PIPE_DEFAULT_OPTIONS

```typescript
// app.module.ts
import { DATE_PIPE_DEFAULT_OPTIONS } from '@angular/common';

@NgModule({
  providers: [
    {
      provide: DATE_PIPE_DEFAULT_OPTIONS,
      useValue: {
        dateFormat: 'dd/MM/yyyy',   // default format everywhere
        timezone: '+0000'           // UTC by default
      }
    }
  ]
})
export class AppModule {}
```

```html
<!-- template — uses global default format -->
<p>Created: {{ createdAt | date }}</p>           <!-- → 27/05/2026 (uses default) -->
<p>Updated: {{ updatedAt | date:'shortTime' }}</p> <!-- → 3:45 PM (override locally) -->
```

---

## Example 7: Modern `polyfills.ts` After Removing IE11

```typescript
// src/polyfills.ts — Angular 13 (clean, no IE11)

/***************************************************
 * Angular is compiled with these TypeScript libs
 ***************************************************/
// Zone.js is the only required polyfill for Angular
import 'zone.js';

// Optional: Web Animations API (for @angular/animations)
// Only needed for Safari < 13.1
// import 'web-animations-js';

/***************************************************
 * APPLICATION IMPORTS
 ***************************************************/
// Add any app-specific polyfills here
```

# Section 3 — Use Cases

---

## Use Case 1: Enterprise Dashboard — Dynamic Widget System

**Problem:** An enterprise dashboard needs to render different widget types (charts, tables, maps, KPIs) based on user configuration. The configuration is stored in a database and loaded at runtime.

**Solution with Angular 13 Dynamic Components:**

```typescript
// widget-registry.ts
import { Type } from '@angular/core';

export type WidgetType = 'chart' | 'table' | 'kpi' | 'map';

export const WIDGET_REGISTRY: Record<WidgetType, () => Promise<Type<any>>> = {
  chart: () => import('./widgets/chart.widget').then(m => m.ChartWidgetComponent),
  table: () => import('./widgets/table.widget').then(m => m.TableWidgetComponent),
  kpi:   () => import('./widgets/kpi.widget').then(m => m.KpiWidgetComponent),
  map:   () => import('./widgets/map.widget').then(m => m.MapWidgetComponent),
};
```

```typescript
// dashboard.component.ts
@Component({
  selector: 'app-dashboard',
  template: `
    <div class="dashboard-grid">
      <div *ngFor="let config of widgetConfigs" class="widget-cell">
        <ng-container #widgetHosts></ng-container>
      </div>
    </div>
  `
})
export class DashboardComponent implements AfterViewInit {
  @ViewChildren('widgetHosts', { read: ViewContainerRef })
  widgetHosts!: QueryList<ViewContainerRef>;

  widgetConfigs: WidgetConfig[] = [];

  constructor(private dashboardService: DashboardService) {}

  async ngAfterViewInit(): Promise<void> {
    this.widgetConfigs = await this.dashboardService.getUserWidgets();
    this.renderWidgets();
  }

  private async renderWidgets(): Promise<void> {
    const hosts = this.widgetHosts.toArray();

    for (let i = 0; i < this.widgetConfigs.length; i++) {
      const config = this.widgetConfigs[i];
      const host = hosts[i];

      // Lazy load the widget class
      const WidgetClass = await WIDGET_REGISTRY[config.type]();

      // Angular 13 — no factory resolver needed!
      const ref = host.createComponent(WidgetClass);
      ref.instance.config = config;
      ref.changeDetectorRef.detectChanges();
    }
  }
}
```

**Benefits:**
- Lazy-loaded widgets: only download widget code when needed
- Config-driven: add new widget types without changing dashboard code
- Clean API: no `ComponentFactoryResolver` boilerplate

---

## Use Case 2: E-Commerce — Faster Builds with Persistent Cache

**Problem:** A large e-commerce Angular app has 200+ components and takes 3 minutes to build. Developers waste time waiting for builds during development. CI/CD pipeline takes 8 minutes per run.

**Solution with Angular 13 Persistent Build Cache:**

```bash
# Local development — first build
$ ng serve
# Initial build: 3 minutes 12 seconds

# Make a change to one component, rebuild
# Without cache: 3 minutes 12 seconds
# With cache:    18 seconds  ← 90% faster!
```

```yaml
# .github/workflows/ci.yml — cache between CI runs
name: Angular CI

on: [push, pull_request]

jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3

      - name: Setup Node.js
        uses: actions/setup-node@v3
        with:
          node-version: '18'
          cache: 'npm'

      - name: Restore Angular Cache
        uses: actions/cache@v3
        with:
          path: .angular/cache
          key: ng-cache-${{ runner.os }}-${{ hashFiles('package-lock.json') }}-${{ github.sha }}
          restore-keys: |
            ng-cache-${{ runner.os }}-${{ hashFiles('package-lock.json') }}-
            ng-cache-${{ runner.os }}-

      - run: npm ci
      - run: ng build --configuration=production

# Result: CI build time reduced from 8 min → 2 min on unchanged modules
```

---

## Use Case 3: SaaS App — Removing IE11 to Use Modern APIs

**Problem:** A SaaS project management app needed to support IE11. This forced the team to avoid CSS Grid, use XMLHttpRequest instead of Fetch, polyfill IntersectionObserver, and maintain 2 build outputs (ES5 + ES2015).

**Solution after dropping IE11 in Angular 13:**

```typescript
// BEFORE — IE11 compatible code
// Intersection Observer polyfill needed
import 'intersection-observer';

// Had to use XMLHttpRequest manually for streaming
// CSS Grid couldn't be used freely
// No CSS Custom Properties

// AFTER Angular 13 — Use modern APIs freely
@Injectable({ providedIn: 'root' })
export class VirtualScrollService {
  observeElement(element: HTMLElement, callback: IntersectionObserverCallback) {
    // No polyfill needed — all modern browsers support this natively
    const observer = new IntersectionObserver(callback, {
      rootMargin: '100px',
      threshold: 0.1
    });
    observer.observe(element);
    return observer;
  }
}
```

```scss
/* Can now freely use CSS Custom Properties (CSS Variables) */
:root {
  --primary: #6200ee;
  --surface: #ffffff;
  --on-surface: #000000;
}

.card {
  background: var(--surface);       /* Works in all supported browsers */
  color: var(--on-surface);
  display: grid;                    /* CSS Grid — no fallback needed */
  grid-template-columns: 1fr 2fr;
}
```

```typescript
// Bundle size comparison
// Before (with IE11 polyfills):   ~450KB gzipped
// After  (Angular 13, no IE11):   ~320KB gzipped  ← 29% smaller!
```

---

## Use Case 4: News Website — Faster Test Runs with TestBed Teardown

**Problem:** A news portal with 500+ component tests was experiencing:
- Tests passing alone but failing when run in sequence
- Memory leaks causing browser to slow down during test runs
- Stale DOM elements from previous tests affecting current test assertions

**Solution with Angular 13 TestBed automatic teardown:**

```typescript
// article-card.component.spec.ts
describe('ArticleCardComponent', () => {
  let fixture: ComponentFixture<ArticleCardComponent>;

  beforeEach(async () => {
    await TestBed.configureTestingModule({
      declarations: [ArticleCardComponent],
      imports: [RouterTestingModule]
      // Angular 13: DOM is destroyed after EACH test automatically
      // No more stale state bleeding between tests!
    }).compileComponents();

    fixture = TestBed.createComponent(ArticleCardComponent);
    fixture.componentInstance.article = mockArticle;
    fixture.detectChanges();
  });

  it('should render article title', () => {
    const h2 = fixture.nativeElement.querySelector('h2');
    expect(h2.textContent).toBe('Breaking News');
    // After this test: DOM is automatically destroyed ✅
  });

  it('should show read time badge', () => {
    // Fresh DOM — no leftovers from previous test ✅
    const badge = fixture.nativeElement.querySelector('.read-time');
    expect(badge.textContent).toContain('5 min read');
  });
});

// Before Angular 13: ~12 minutes for 500 tests (memory leaks slow things down)
// After Angular 13:  ~7 minutes for 500 tests (clean state, no leaks)
```

---

## Use Case 5: Financial App — TypeScript 4.4 Strict Error Handling

**Problem:** A financial trading app had untyped `catch` blocks, causing different error types to be handled the same way and masking bugs.

**Solution using TypeScript 4.4's `unknown` in catch blocks:**

```typescript
// trading.service.ts
@Injectable({ providedIn: 'root' })
export class TradingService {
  async executeTrade(order: TradeOrder): Promise<TradeResult> {
    try {
      const result = await this.api.post<TradeResult>('/trades', order);
      return result;
    } catch (err: unknown) {  // TypeScript 4.4 — err is unknown, not any

      // Must narrow the type before using it
      if (err instanceof NetworkError) {
        this.alertService.warn('Network issue — please retry');
        throw err;
      }

      if (err instanceof InsufficientFundsError) {
        this.alertService.error(`Insufficient funds: need ${err.required}, have ${err.available}`);
        throw err;
      }

      if (err instanceof HttpErrorResponse) {
        if (err.status === 429) {
          this.alertService.warn('Rate limited — retry in 30 seconds');
        } else {
          this.alertService.error(`Server error: ${err.error.message}`);
        }
        throw err;
      }

      // Fallback for unexpected errors
      console.error('Unexpected error:', err);
      throw new Error('An unexpected error occurred');
    }
  }
}
```

---

## Use Case Summary Table

| Use Case | Feature Used | Business Value |
|---|---|---|
| Enterprise Dashboard | Dynamic components (new API) | Modular, lazy-loaded widgets — clean code |
| E-Commerce CI/CD | Persistent build cache | 75% faster CI builds — save developer time |
| SaaS App | No IE11 | 29% smaller bundle, modern CSS/JS APIs |
| News Portal Tests | TestBed teardown | Reliable tests, 40% faster test suite |
| Financial Trading | TypeScript 4.4 | Type-safe error handling — prevents runtime bugs |

# Section 4 — Interview Q&A

---

## Basic Level Questions

---

### Q1. What is Angular 13 and when was it released?
**Answer:**
Angular 13 was released on **November 3, 2021**. It is a major version that permanently removed the old View Engine and made **Ivy the only rendering and compilation engine**. It also dropped IE11 support, upgraded to TypeScript 4.4 and RxJS 7.4, and introduced persistent build cache.

---

### Q2. What is the difference between View Engine and Ivy?

| Aspect | View Engine | Ivy |
|---|---|---|
| Compilation | Global — needs full context | Local — each component compiled independently |
| Bundle size | Larger | Smaller (tree-shakable) |
| Rebuild speed | Slower | Faster |
| Debug tools | Limited | `ng.getComponent()`, `ng.applyChanges()` |
| Template type-checking | Basic | Strict, full type-checking |
| Status in Angular 13 | ❌ Removed | ✅ Only engine |

**Answer:**
View Engine was Angular's original compilation pipeline that required global knowledge of all components to compile any single component. Ivy replaced it with a locality-based approach where each component is compiled independently, enabling smaller bundles, faster builds, and better debugging.

---

### Q3. Why did Angular 13 drop IE11 support?

**Answer:**
Angular 13 dropped IE11 support because:
1. IE11's global market share had fallen below **1%**
2. Supporting IE11 required shipping **two builds** (ES5 for IE11, ES2015+ for modern browsers) — called "differential loading"
3. IE11 polyfills added **~50KB** to the bundle size
4. Removing IE11 allows Angular to target **ES2017+**, enabling use of modern JavaScript and browser APIs (Fetch API, CSS Grid, CSS Variables, IntersectionObserver) without polyfills

---

### Q4. How do you upgrade a project from Angular 12 to Angular 13?

**Answer:**
```bash
# 1. Update Angular core and CLI
ng update @angular/core@13 @angular/cli@13

# 2. Update Angular Material (if used)
ng update @angular/material@13

# 3. Update RxJS (if not already on 7.x)
ng update rxjs@7

# 4. Clean up polyfills.ts — remove IE11 polyfills
# Remove: core-js/es/*, classlist.js, web-animations-js

# 5. Remove IE11 from browserslist
# In .browserslistrc, remove: IE 11

# 6. Update tsconfig.json target from ES5 to ES2017
```

---

### Q5. What is `ComponentFactoryResolver` and is it still needed in Angular 13?

**Answer:**
`ComponentFactoryResolver` was a service used to get a **ComponentFactory** — an object that knew how to create a specific component dynamically. In Angular 13 it is **deprecated** but still present for backward compatibility.

In Angular 13+, you can pass the component **class directly** to `ViewContainerRef.createComponent()`:
```typescript
// Angular 13 — no factory resolver needed
const ref = this.viewContainerRef.createComponent(MyComponent);
```
It will be **removed in a future major version**.

---

## Intermediate Level Questions

---

### Q6. Explain the Angular 13 Persistent Build Cache. How does it work?

**Answer:**
Angular CLI 13 introduced disk-based caching enabled by default. When you run `ng build` or `ng serve`:
1. Angular hashes each file and its dependencies
2. Compiled outputs are stored in `.angular/cache/`
3. On the next build, unchanged files are restored from cache
4. Only modified files and their direct dependents are recompiled

**Configuration in `angular.json`:**
```json
{
  "cli": {
    "cache": {
      "enabled": true,
      "path": ".angular/cache",
      "environment": "all"   // "local" | "ci" | "all"
    }
  }
}
```

**Typical improvement:** 40–90% faster incremental builds. The `.angular/cache` folder should be added to `.gitignore`.

---

### Q7. What changes did Angular 13 bring to TestBed?

**Answer:**
Angular 13 changed the default **teardown behavior** of `TestBed`. Previously, `TestBed` would keep the testing module and DOM alive between tests, which caused:
- Memory leaks as the test suite grew
- Test pollution (stale DOM from one test affecting the next)

In Angular 13, `TestBed` **automatically destroys the testing module and removes the DOM after each test** (`destroyAfterEach: true` is now the default).

To opt out during migration:
```typescript
TestBed.configureTestingModule({
  teardown: { destroyAfterEach: false }
});
```

---

### Q8. What is the Angular Package Format (APF) v13 and why does it matter for library authors?

**Answer:**
APF is the specification for how Angular libraries should be packaged and published to npm. APF v13 makes these changes:
- **Removed UMD bundles** (no longer needed since bundlers handle this)
- **Removed View Engine metadata** (`.metadata.json` files) — only Ivy is supported
- Libraries must ship **ES2020 + ESM** (ECMAScript modules)
- **Partial compilation** is required for libraries to be compatible with different Angular versions

```json
// tsconfig.lib.prod.json in Angular library
{
  "angularCompilerOptions": {
    "compilationMode": "partial"  // required for library authors in v13+
  }
}
```

---

### Q9. What is `firstValueFrom` and `lastValueFrom` in RxJS 7? Why were they introduced?

**Answer:**
These are replacements for the deprecated `.toPromise()` method:

- **`firstValueFrom(obs$)`** — returns a Promise that resolves with the **first emitted value**, then unsubscribes. Throws `EmptyError` if the observable completes without emitting.
- **`lastValueFrom(obs$)`** — returns a Promise that resolves with the **last emitted value** when the observable completes. Throws `EmptyError` if empty.

They were introduced because `.toPromise()` had an ambiguous behavior — it resolved with `undefined` for empty observables (a silent failure). The new APIs throw explicitly, making errors visible.

```typescript
// OLD — ambiguous
const val = await obs$.toPromise();  // undefined if empty, no error

// NEW — explicit behavior
const val = await firstValueFrom(obs$);  // throws EmptyError if empty
const val = await lastValueFrom(obs$, { defaultValue: [] });  // or provide default
```

---

### Q10. How does TypeScript 4.4's control flow analysis of aliased conditions work?

**Answer:**
Before TypeScript 4.4, narrowing only worked on the condition directly in an `if` statement. TypeScript 4.4 can now track type narrowing through **aliased boolean expressions**:

```typescript
// TypeScript < 4.4 — narrowing lost
const isString = typeof value === 'string';
if (isString) {
  value.toUpperCase();  // ❌ Error: value might not be string
}

// TypeScript 4.4 — narrowing works through alias
const isString = typeof value === 'string';
if (isString) {
  value.toUpperCase();  // ✅ TypeScript knows value is string here
}
```

---

## Advanced Level Questions

---

### Q11. Explain Ivy's "locality principle" and why it matters.

**Answer:**
The locality principle means each Angular component is compiled **using only its own file** — no information from parent modules, sibling components, or the application structure is needed.

**Why it matters:**
1. **Incremental compilation**: If you change `ButtonComponent`, only `ButtonComponent` needs recompiling — not all components that use it.
2. **Library isolation**: Libraries can be pre-compiled without knowing which app will use them.
3. **Faster CI builds**: Changed files trigger minimal recompilation.

**Under the hood:** Ivy compiles each component to a set of **template instructions** (functions like `ɵɵelement`, `ɵɵtext`, `ɵɵproperty`) that directly manipulate the DOM, rather than using a virtual DOM diffing algorithm.

---

### Q12. What is the `createComponent` function introduced in Angular 13 and how does it differ from `ViewContainerRef.createComponent()`?

**Answer:**
Angular 13 introduced two ways to create dynamic components:

**`ViewContainerRef.createComponent(Type)`** — creates a component inside a specific location in the DOM (in a `ng-container` or host element):
```typescript
// Attached to a ViewContainerRef — part of the Angular component tree
const ref = this.vcr.createComponent(MyComponent);
```

**`createComponent(Type, { environmentInjector })`** — creates a component **outside** the normal component tree, useful for modals and toasts that need to be appended to `document.body`:
```typescript
import { createComponent } from '@angular/core';
const ref = createComponent(MyComponent, {
  environmentInjector: this.envInjector
});
this.appRef.attachView(ref.hostView);
document.body.appendChild(ref.location.nativeElement);
```

---

### Q13. How does differential loading work and why was it removed in Angular 13?

**Answer:**
**Differential loading** was introduced in Angular 8 to serve **two separate JavaScript bundles**:
- `main-es2015.js` — for modern browsers (Chrome, Firefox, Safari)
- `main-es5.js` — for IE11

The correct bundle was selected at runtime using `<script type="module">` (modern browsers) and `<script nomodule>` (IE11).

**Removed in Angular 13 because:**
1. IE11 support was dropped — ES5 bundle no longer needed
2. Eliminated the need to build two bundles (saves build time)
3. All supported browsers handle ES2017+ natively

**Impact:** Build process is simpler and faster. Bundle sizes are reduced because ES2017 syntax (async/await, etc.) is more compact than transpiled ES5.

---

### Q14. What is partial compilation in Angular and when should library authors use it?

**Answer:**
Partial compilation (`compilationMode: 'partial'`) produces **intermediate compiled output** (called "declaration" format) that is not fully compiled. When the consuming application builds, it **finishes the compilation** with the app's specific Angular version.

**Why it's needed:** If a library uses full Ivy compilation for Angular 14, it might not be compatible with an app using Angular 13 (due to internal API differences).

**Partial compilation is the middle ground:**
- Library ships: partially compiled (stable, version-agnostic output)
- App compiles: finishes compilation using the app's own Angular version

```json
// tsconfig.lib.prod.json
{
  "angularCompilerOptions": {
    "compilationMode": "partial"  // Required for npm-published libraries
  }
}
```

`ng-packagr` (used by `ng generate library`) handles this automatically.

---

### Q15. How would you configure Angular 13 build cache for a CI/CD environment with multiple parallel jobs?

**Answer:**
```yaml
# .github/workflows/build.yml
name: CI

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3

      - name: Cache node modules
        uses: actions/cache@v3
        with:
          path: node_modules
          key: npm-${{ hashFiles('package-lock.json') }}

      - name: Cache Angular build
        uses: actions/cache@v3
        with:
          path: .angular/cache
          # Include branch name so parallel branches don't share cache
          key: ng-${{ runner.os }}-${{ github.ref }}-${{ hashFiles('src/**/*.ts') }}
          restore-keys: |
            ng-${{ runner.os }}-${{ github.ref }}-
            ng-${{ runner.os }}-refs/heads/main-

      - run: npm ci
      - run: ng test --watch=false
      - run: ng build --configuration=production
```

**Configure cache to only enable in CI:**
```json
{
  "cli": {
    "cache": {
      "environment": "ci"   // only cache in CI, skip locally (or "all" for both)
    }
  }
}
```

---

## Scenario-Based Questions

---

### Q16. Your Angular 12 app uses `ComponentFactoryResolver` in 15 places. How would you migrate to Angular 13?

**Answer:**
1. Run `ng update @angular/core@13` — the Angular schematics may auto-migrate some cases
2. Find all usages: `grep -r "ComponentFactoryResolver" src/`
3. For each usage, replace the pattern:

```typescript
// FIND this pattern:
constructor(private resolver: ComponentFactoryResolver, private vcr: ViewContainerRef) {}
const factory = this.resolver.resolveComponentFactory(MyComp);
const ref = this.vcr.createComponent(factory);

// REPLACE with:
constructor(private vcr: ViewContainerRef) {}
const ref = this.vcr.createComponent(MyComp);
```

4. Remove `ComponentFactoryResolver` from constructor injections
5. Remove `entryComponents` from `@NgModule` (also deprecated/removed in v13)

---

### Q17. After upgrading to Angular 13, your unit tests start failing with "Cannot read properties of null". What is likely the cause and how do you fix it?

**Answer:**
The cause is the **new TestBed teardown behavior**. Angular 13 destroys the DOM after each test, so if your `beforeEach` or `afterEach` hooks reference DOM elements that were destroyed, they'll throw null errors.

**Fix option 1 — Update tests to not rely on stale references:**
```typescript
// BAD — caches DOM reference across tests
let button: HTMLElement;
beforeEach(() => {
  button = fixture.nativeElement.querySelector('button');  // cached!
});
afterEach(() => {
  button.click();  // ❌ button might be destroyed
});

// GOOD — query DOM fresh in each test
it('should handle click', () => {
  const button = fixture.nativeElement.querySelector('button');  // fresh query
  button.click();
  // ...
});
```

**Fix option 2 — Temporarily opt out:**
```typescript
TestBed.configureTestingModule({
  teardown: { destroyAfterEach: false }  // opt out temporarily
});
```

---

### Q18. A team member says "we should keep the .angular/cache folder in Git to share build cache". Is this correct?

**Answer:**
**No, this is incorrect.** The `.angular/cache` folder should be in `.gitignore` because:
1. It contains machine-specific binary files that don't compress well in Git
2. Cache is keyed by file hashes — different machines will regenerate it anyway
3. It can grow large (hundreds of MB)
4. Sharing it via Git provides no benefit since each machine generates its own cache

**Correct approach for sharing cache in CI:** Use the CI platform's cache mechanism (GitHub Actions `cache` action, GitLab CI `cache` keyword, etc.) keyed on `package-lock.json` hash.

```gitignore
# .gitignore
.angular/cache
```

---

### Q19. How does removing IE11 in Angular 13 affect existing CSS in your app?

**Answer:**
Removing IE11 opens up modern CSS features freely:

**Before (IE11 workarounds needed):**
```scss
// Had to use float-based layouts
.grid { display: flex; flex-wrap: wrap; }
.item { float: left; width: 33.33%; }

// No CSS Custom Properties
.primary { background-color: #6200ee; }

// No CSS Grid
```

**After Angular 13 (modern CSS):**
```scss
// CSS Grid freely usable
.grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 16px; }

// CSS Custom Properties
:root { --primary: #6200ee; }
.primary { background-color: var(--primary); }

// :is() and :where() pseudo-classes
:is(h1, h2, h3):hover { color: var(--primary); }
```

**Also safe to use without polyfills:**
- `IntersectionObserver`
- `ResizeObserver`
- CSS `aspect-ratio`
- `Array.prototype.flat()`
- `Object.fromEntries()`

---

### Q20. What are `entryComponents` and why were they removed in Angular 13?

**Answer:**
`entryComponents` was a property in `@NgModule` that told Angular which components would be created dynamically (not via a template selector), so their factories would be included in the bundle.

**Before Ivy (View Engine):** Angular needed to know about all dynamically created components upfront to compile their factories. `entryComponents` told the compiler "include factory for these components even if they're not used in a template."

**With Ivy:** There are no factories. Every component is self-describing and tree-shakable. The compiler can determine which components need to be included automatically.

```typescript
// Angular < 13 (View Engine) — entryComponents needed
@NgModule({
  declarations: [AppComponent, ModalComponent],
  entryComponents: [ModalComponent],  // needed for dynamic creation
})
export class AppModule {}

// Angular 13 (Ivy) — entryComponents is IGNORED (deprecated and removed)
@NgModule({
  declarations: [AppComponent, ModalComponent],
  // No entryComponents needed!
})
export class AppModule {}
```

---

## Quick Reference Card

### Angular 13 Key Facts for Interviews

| Question | Answer |
|---|---|
| Release date | November 3, 2021 |
| Rendering engine | Ivy only (View Engine removed) |
| IE11 | Dropped |
| TypeScript version | 4.4.x |
| RxJS version | 7.4+ |
| Build cache | Enabled by default (`.angular/cache`) |
| Dynamic components | `createComponent(Class)` — no factory resolver |
| TestBed teardown | `destroyAfterEach: true` is now default |
| entryComponents | Deprecated and ignored |
| Bundle size improvement | ~25% smaller (no IE11 polyfills, ES2017 target) |
| Compilation | Locality-based (each component independently) |
| Library format | APF v13 — ES2020+ESM, no UMD |

### Comparison Cheatsheet

```
View Engine  →  Ivy
entryComponents  →  (removed, not needed)
ComponentFactoryResolver  →  ViewContainerRef.createComponent(Class)
.toPromise()  →  firstValueFrom() / lastValueFrom()
teardown: false  →  teardown: true (default)
ES5 target  →  ES2017 target
2 builds (differential loading)  →  1 build
~3min rebuild  →  ~18sec rebuild (with cache)
```